In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [2]:
X = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_road_3.csv')
S = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_speed_3.csv')
y = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\y_train_3.csv')

In [3]:
X_train = torch.tensor(X.values, dtype=torch.float32).to(device='cuda')
S_train = torch.tensor(S.values, dtype=torch.float32).to(device='cuda')
y_tensor = torch.tensor(y.values, dtype=torch.float32).to(device='cuda')

In [4]:
print(len(X_train))
print(len(S_train))
print(len(y_tensor))

30650
30650
30650


In [5]:
X_tensor = torch.cat((X_train, S_train), dim=1)
print(X_tensor[0])
print(X_tensor[0].size())

tensor([0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.4882], device='cuda:0')
torch.Size([12289])


In [6]:
print(y_tensor[0])
print(y_tensor[0].size())

tensor([-0.0044, -0.0028], device='cuda:0')
torch.Size([2])


In [7]:
dataset = TensorDataset(X_tensor, y_tensor)
data_loader = DataLoader(dataset, batch_size=512, shuffle=False)

In [8]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=12289, hidden_size=512, hidden_size_1=512, output_size=2, num_layers=3):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)

        self.linear = nn.Linear(hidden_size, hidden_size_1)
        self.fc2 = nn.Linear(hidden_size_1, output_size)
        self.relu = nn.ReLU()
        self.tanh = nn.Tanh()

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).cuda()
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).cuda()

        out, _ = self.lstm(x, (h0, c0))

        out = out[:, -1, :]

        out = self.linear(out)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.tanh(out)
        return out.squeeze(0)

In [9]:
model = LSTMModel().cuda()

In [10]:
# Функция для расчета L1 штрафа
def l1_penalty(params):
    return sum(p.abs().sum() for p in params)

# Функция для расчета L2 штрафа
def l2_penalty(params):
    return sum(p.pow(2.0).sum() for p in params)

# Обучение модели с L1 и L2 регуляризацией
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0000001, weight_decay=0.0001)
l1_factor = 1e-5
l2_factor = 1e-5


In [11]:
# Обучение модели
num_epochs = 50
for epoch in range(num_epochs):
    for batch_x, batch_y in data_loader:
        optimizer.zero_grad()

        outputs = model(batch_x.unsqueeze(0))
        loss = criterion(outputs, batch_y)
        
        l1_loss = l1_penalty(model.parameters())
        l2_loss = l2_penalty(model.parameters())
        total_loss = loss + l1_factor * l1_loss + l2_factor * l2_loss
        
        total_loss.backward()
        optimizer.step()

    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.9f}')

C:\Users\filip\anaconda3\envs\ETS_autopylot\Lib\site-packages\torch\nn\modules\loss.py:535: UserWarning: Using a target size (torch.Size([512, 2])) that is different to the input size (torch.Size([2])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
C:\Users\filip\anaconda3\envs\ETS_autopylot\Lib\site-packages\torch\nn\modules\loss.py:535: UserWarning: Using a target size (torch.Size([442, 2])) that is different to the input size (torch.Size([2])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch [1/50], Loss: 0.000398733
Epoch [2/50], Loss: 0.000391832
Epoch [3/50], Loss: 0.000385069
Epoch [4/50], Loss: 0.000378442
Epoch [5/50], Loss: 0.000371946
Epoch [6/50], Loss: 0.000365585
Epoch [7/50], Loss: 0.000359346
Epoch [8/50], Loss: 0.000353241
Epoch [9/50], Loss: 0.000347235
Epoch [10/50], Loss: 0.000341360
Epoch [11/50], Loss: 0.000335615
Epoch [12/50], Loss: 0.000329983
Epoch [13/50], Loss: 0.000324460
Epoch [14/50], Loss: 0.000319046
Epoch [15/50], Loss: 0.000313740
Epoch [16/50], Loss: 0.000308540
Epoch [17/50], Loss: 0.000303446
Epoch [18/50], Loss: 0.000298452
Epoch [19/50], Loss: 0.000293557
Epoch [20/50], Loss: 0.000288756
Epoch [21/50], Loss: 0.000284053
Epoch [22/50], Loss: 0.000279443
Epoch [23/50], Loss: 0.000274923
Epoch [24/50], Loss: 0.000270493
Epoch [25/50], Loss: 0.000266149
Epoch [26/50], Loss: 0.000261887
Epoch [27/50], Loss: 0.000257712
Epoch [28/50], Loss: 0.000253620
Epoch [29/50], Loss: 0.000249612
Epoch [30/50], Loss: 0.000245688
Epoch [31/50], Loss

In [12]:
torch.save(model.state_dict(), 'C:\PycharmProjects\ETS_Autopilot\static\weight_model\weight_wheel_nn_lstm_5_3.pth')